# 03_FEATURE ENGINEERING

## Objective

Create domain-inspired financial ratios and aggregate bureau information to improve predictive performance.

## Inputs

Application dataset

Bureau dataset

## Outputs

Feature-engineered modelling dataset

### 1. Imports and load data

In [135]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

app = pd.read_csv('application_train.csv', nrows=50000)

bureau = pd.read_csv('bureau.csv', nrows=100000)

print("Application Shape:", app.shape)
print("Bureau Shape:", bureau.shape)

Application Shape: (50000, 122)
Bureau Shape: (100000, 17)


### 2. Important sampling problem

In [136]:
app_ids = set(app['SK_ID_CURR'])

bureau_ids = set(bureau['SK_ID_CURR'])

common_ids = app_ids.intersection(bureau_ids)

print("Application customers:", len(app_ids))
print("Bureau customers:", len(bureau_ids))
print("Common customers:", len(common_ids))
print(
    "Application customers with bureau records:",
    round(len(common_ids) / len(app_ids) * 100, 2),
    "%"
)

Application customers: 50000
Bureau customers: 23121
Common customers: 3440
Application customers with bureau records: 6.88 %


In [137]:
app = pd.read_csv('application_train.csv', nrows=50000)

app_ids = set(app['SK_ID_CURR'])

bureau_chunks = []

for chunk in pd.read_csv('bureau.csv', chunksize=100000):
    matched_rows = chunk[
        chunk['SK_ID_CURR'].isin(app_ids)
    ]

    bureau_chunks.append(matched_rows)

bureau = pd.concat(
    bureau_chunks,
    ignore_index=True
)

print("Application Shape:", app.shape)
print("Filtered Bureau Shape:", bureau.shape)
print(
    "Application Customers with Bureau Records:",
    bureau['SK_ID_CURR'].nunique()
)

Application Shape: (50000, 122)
Filtered Bureau Shape: (238223, 17)
Application Customers with Bureau Records: 42851


### 3. Clean known anomalous values

In [138]:
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(
    365243,
    np.nan
)

### 4. Create application-level features

#### Age

In [139]:
app['AGE'] = (
    app['DAYS_BIRTH'].abs() / 365.25
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\4282977115.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['AGE'] = (


#### Employment years

In [140]:
app['EMPLOYMENT_YEARS'] = (
    app['DAYS_EMPLOYED'].abs() / 365.25
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\3788055926.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['EMPLOYMENT_YEARS'] = (


#### Credit-to-income ratio

In [141]:
app['CREDIT_INCOME_RATIO'] = (
    app['AMT_CREDIT']
    / app['AMT_INCOME_TOTAL']
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\111055340.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['CREDIT_INCOME_RATIO'] = (


#### Annuity-to-income ratio

In [142]:
app['ANNUITY_INCOME_RATIO'] = (
    app['AMT_ANNUITY']
    / app['AMT_INCOME_TOTAL']
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\2930268624.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['ANNUITY_INCOME_RATIO'] = (


#### Credit-to-annuity ratio

In [143]:
app['CREDIT_ANNUITY_RATIO'] = (
    app['AMT_CREDIT']
    / app['AMT_ANNUITY']
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\940994623.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['CREDIT_ANNUITY_RATIO'] = (


#### Income per family member

In [144]:
app['INCOME_PER_PERSON'] = (
    app['AMT_INCOME_TOTAL']
    / app['CNT_FAM_MEMBERS']
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\2193157967.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['INCOME_PER_PERSON'] = (


#### Employment-to-age ratio

In [145]:
app['EMPLOYMENT_AGE_RATIO'] = (
    app['EMPLOYMENT_YEARS']
    / app['AGE']
)

C:\Users\rishi\AppData\Local\Temp\ipykernel_3624\3276571370.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app['EMPLOYMENT_AGE_RATIO'] = (


#### 5. Validate the engineered features

In [146]:
application_features = [
    'AGE',
    'EMPLOYMENT_YEARS',
    'CREDIT_INCOME_RATIO',
    'ANNUITY_INCOME_RATIO',
    'CREDIT_ANNUITY_RATIO',
    'INCOME_PER_PERSON',
    'EMPLOYMENT_AGE_RATIO'
]

app[application_features].describe(
    percentiles=[0.01, 0.5, 0.95, 0.99]
).T

,count,mean,std,min,1%,50%,95%,99%,max
AGE,50000.0,43.866147,11.940816,21.026694,22.570842,43.069131,63.509925,66.874771,6.895003e+01
EMPLOYMENT_YEARS,41076.0,6.566773,6.442337,0.000000,0.298426,4.531143,20.203285,31.130732,4.799726e+01
CREDIT_INCOME_RATIO,50000.0,3.962126,2.713134,0.004808,0.586298,3.288356,9.101595,13.084854,8.473684e+01
ANNUITY_INCOME_RATIO,49999.0,0.181018,0.095025,0.000224,0.039500,0.163075,0.354503,0.482901,1.875965e+00
CREDIT_ANNUITY_RATIO,49999.0,21.642400,7.831805,8.036739,8.733611,20.000000,34.565996,37.788815,4.529553e+01
INCOME_PER_PERSON,49999.0,93379.744532,187345.136423,7200.000000,16875.000000,75000.000000,225000.000000,341118.000000,3.900000e+07
EMPLOYMENT_AGE_RATIO,41076.0,0.157772,0.134494,0.000000,0.007492,0.118926,0.446621,0.592533,7.228086e-01


In [147]:
np.isinf(
    app[application_features]
).sum()

AGE                     0
EMPLOYMENT_YEARS        0
CREDIT_INCOME_RATIO     0
ANNUITY_INCOME_RATIO    0
CREDIT_ANNUITY_RATIO    0
INCOME_PER_PERSON       0
EMPLOYMENT_AGE_RATIO    0
dtype: int64

In [148]:
app[
    application_features
].isnull().mean().mul(100).sort_values(
    ascending=False
)

EMPLOYMENT_YEARS        17.848
EMPLOYMENT_AGE_RATIO    17.848
ANNUITY_INCOME_RATIO     0.002
CREDIT_ANNUITY_RATIO     0.002
INCOME_PER_PERSON        0.002
CREDIT_INCOME_RATIO      0.000
AGE                      0.000
dtype: float64

### 6. Understand bureau before aggregation

In [149]:
bureau_features_to_inspect = [
    'CREDIT_ACTIVE',
    'CREDIT_TYPE',
    'DAYS_CREDIT',
    'CREDIT_DAY_OVERDUE',
    'DAYS_CREDIT_ENDDATE',
    'AMT_CREDIT_MAX_OVERDUE',
    'CNT_CREDIT_PROLONG',
    'AMT_CREDIT_SUM',
    'AMT_CREDIT_SUM_DEBT',
    'AMT_CREDIT_SUM_OVERDUE'
]

bureau[
    bureau_features_to_inspect
].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CREDIT_ACTIVE,238223,3,Closed,149022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CREDIT_TYPE,238223,13,Consumer credit,174032,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DAYS_CREDIT,238223.0,NaN,NaN,NaN,-1146.402417,801.833369,-2922.0,-1675.0,-988.0,-471.0,0.0
CREDIT_DAY_OVERDUE,238223.0,NaN,NaN,NaN,0.820362,36.309586,0.0,0.0,0.0,0.0,2679.0
DAYS_CREDIT_ENDDATE,224009.0,NaN,NaN,NaN,493.148351,4967.089076,-42042.0,-1149.0,-330.0,477.0,31198.0
AMT_CREDIT_MAX_OVERDUE,84322.0,NaN,NaN,NaN,3493.435616,47455.437597,0.0,0.0,0.0,0.0,5487205.5
CNT_CREDIT_PROLONG,238223.0,NaN,NaN,NaN,0.007409,0.103379,0.0,0.0,0.0,0.0,8.0
AMT_CREDIT_SUM,238222.0,NaN,NaN,NaN,353591.92702,1017694.185098,0.0,51300.0,124776.0,310500.0,142290000.0
AMT_CREDIT_SUM_DEBT,201515.0,NaN,NaN,NaN,137510.689111,667834.722449,-2167229.34,0.0,0.0,39264.8625,64570243.5
AMT_CREDIT_SUM_OVERDUE,238223.0,NaN,NaN,NaN,46.335901,6300.912088,0.0,0.0,0.0,0.0,2387232.0


In [150]:
bureau['CREDIT_ACTIVE'].value_counts(
    dropna=False
)

CREDIT_ACTIVE
Closed    149022
Active     88205
Sold         996
Name: count, dtype: int64

In [151]:
bureau['CREDIT_TYPE'].value_counts(
    dropna=False
)

CREDIT_TYPE
Consumer credit                           174032
Credit card                                55760
Car loan                                    3792
Mortgage                                    2594
Microloan                                   1503
Loan for business development                263
Another type of loan                         131
Unknown type of loan                          75
Loan for working capital replenishment        65
Real estate loan                               4
Cash loan (non-earmarked)                      2
Loan for the purchase of equipment             1
Interbank credit                               1
Name: count, dtype: int64

### 7. Create helper indicators in bureau
#### Active credit indicator

In [152]:
bureau['IS_ACTIVE'] = (
    bureau['CREDIT_ACTIVE'] == 'Active'
).astype(int)

#### Closed credit indicator

In [153]:
bureau['IS_CLOSED'] = (
    bureau['CREDIT_ACTIVE'] == 'Closed'
).astype(int)

#### Overdue credit indicator

In [154]:
bureau['HAS_OVERDUE'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE']
    > 0
).astype(int)

#### Has debt indicator

In [155]:
bureau['HAS_DEBT'] = (
    bureau['AMT_CREDIT_SUM_DEBT']
    > 0
).astype(int)

### 8. Aggregate bureau to customer level

In [156]:
bureau_agg = (
    bureau
    .groupby('SK_ID_CURR')
    .agg(
        BUREAU_LOAN_COUNT=(
            'SK_ID_BUREAU',
            'count'
        ),

        BUREAU_ACTIVE_LOAN_COUNT=(
            'IS_ACTIVE',
            'sum'
        ),

        BUREAU_CLOSED_LOAN_COUNT=(
            'IS_CLOSED',
            'sum'
        ),

        BUREAU_OVERDUE_LOAN_COUNT=(
            'HAS_OVERDUE',
            'sum'
        ),

        BUREAU_DEBT_LOAN_COUNT=(
            'HAS_DEBT',
            'sum'
        ),

        BUREAU_MEAN_CREDIT=(
            'AMT_CREDIT_SUM',
            'mean'
        ),

        BUREAU_MAX_CREDIT=(
            'AMT_CREDIT_SUM',
            'max'
        ),

        BUREAU_TOTAL_CREDIT=(
            'AMT_CREDIT_SUM',
            'sum'
        ),

        BUREAU_TOTAL_DEBT=(
            'AMT_CREDIT_SUM_DEBT',
            'sum'
        ),

        BUREAU_MEAN_DAYS_CREDIT=(
            'DAYS_CREDIT',
            'mean'
        ),

        BUREAU_MAX_DAYS_OVERDUE=(
            'CREDIT_DAY_OVERDUE',
            'max'
        ),

        BUREAU_CREDIT_PROLONG_COUNT=(
            'CNT_CREDIT_PROLONG',
            'sum'
        )
    )
    .reset_index()
)

bureau_agg.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_LOAN_COUNT,BUREAU_CLOSED_LOAN_COUNT,BUREAU_OVERDUE_LOAN_COUNT,BUREAU_DEBT_LOAN_COUNT,BUREAU_MEAN_CREDIT,BUREAU_MAX_CREDIT,BUREAU_TOTAL_CREDIT,BUREAU_TOTAL_DEBT,BUREAU_MEAN_DAYS_CREDIT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_CREDIT_PROLONG_COUNT
0,100002,8,2,6,0,1,108131.945625,450000.0,865055.565,245781.0,-874.000000,0,0
1,100003,4,1,3,0,0,254350.125000,810000.0,1017400.500,0.0,-1400.750000,0,0
2,100004,2,0,2,0,0,94518.900000,94537.8,189037.800,0.0,-867.000000,0,0
3,100007,1,0,1,0,0,146250.000000,146250.0,146250.000,0.0,-1149.000000,0,0
4,100008,3,1,2,0,1,156148.500000,267606.0,468445.500,240057.0,-757.333333,0,0


In [157]:
bureau_agg.shape

(42851, 13)

In [158]:
bureau_agg['SK_ID_CURR'].is_unique

True

### 9. Create bureau ratio features
#### Active-loan ratio

In [159]:
bureau_agg['BUREAU_ACTIVE_LOAN_RATIO'] = (
    bureau_agg['BUREAU_ACTIVE_LOAN_COUNT']
    / bureau_agg['BUREAU_LOAN_COUNT']
)

#### Overdue-loan ratio

In [160]:
bureau_agg['BUREAU_OVERDUE_LOAN_RATIO'] = (
    bureau_agg['BUREAU_OVERDUE_LOAN_COUNT']
    / bureau_agg['BUREAU_LOAN_COUNT']
)

#### Debt-to-credit ratio

In [161]:
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = (
    bureau_agg['BUREAU_TOTAL_DEBT'] 
    / bureau_agg['BUREAU_TOTAL_CREDIT']
)

In [162]:
bureau_agg.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_LOAN_COUNT,BUREAU_CLOSED_LOAN_COUNT,BUREAU_OVERDUE_LOAN_COUNT,BUREAU_DEBT_LOAN_COUNT,BUREAU_MEAN_CREDIT,BUREAU_MAX_CREDIT,BUREAU_TOTAL_CREDIT,BUREAU_TOTAL_DEBT,BUREAU_MEAN_DAYS_CREDIT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_CREDIT_PROLONG_COUNT,BUREAU_ACTIVE_LOAN_RATIO,BUREAU_OVERDUE_LOAN_RATIO,BUREAU_DEBT_CREDIT_RATIO
0,100002,8,2,6,0,1,108131.945625,450000.0,865055.565,245781.000,-874.000000,0,0,0.250000,0.0,0.284122
1,100003,4,1,3,0,0,254350.125000,810000.0,1017400.500,0.000,-1400.750000,0,0,0.250000,0.0,0.000000
2,100004,2,0,2,0,0,94518.900000,94537.8,189037.800,0.000,-867.000000,0,0,0.000000,0.0,0.000000
3,100007,1,0,1,0,0,146250.000000,146250.0,146250.000,0.000,-1149.000000,0,0,0.000000,0.0,0.000000
4,100008,3,1,2,0,1,156148.500000,267606.0,468445.500,240057.000,-757.333333,0,0,0.333333,0.0,0.512454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42846,157872,1,0,1,0,0,324000.000000,324000.0,324000.000,0.000,-657.000000,0,0,0.000000,0.0,0.000000
42847,157873,4,1,3,0,1,202500.000000,315000.0,810000.000,174946.500,-1431.000000,0,2,0.250000,0.0,0.215983
42848,157874,23,4,19,0,4,108087.343043,557158.5,2486008.890,843796.755,-412.782609,0,0,0.173913,0.0,0.339418
42849,157875,8,2,6,0,1,321566.023125,1436170.5,2572528.185,1370755.440,-1419.000000,0,0,0.250000,0.0,0.532844


### 10. Merge application and bureau features


In [163]:
print("Application rows:", len(app))
print("Bureau aggregate rows:", len(bureau_agg))

Application rows: 50000
Bureau aggregate rows: 42851


In [164]:
model_data = app.merge(
    bureau_agg,
    on='SK_ID_CURR',
    how='left',
    validate='one_to_one'
)

In [165]:
# validate
print("Application Shape:", app.shape)
print("Final Dataset Shape:", model_data.shape)

assert len(model_data) == len(app)

print("Merge validation passed.")

Application Shape: (50000, 129)
Final Dataset Shape: (50000, 144)
Merge validation passed.


### 11. Examine customers without bureau history

In [166]:
no_bureau_percentage = (
    model_data['BUREAU_LOAN_COUNT']
    .isnull()
    .mean()
    * 100
)

print(
    "Customers without bureau records:",
    round(no_bureau_percentage, 2),
    "%"
)

Customers without bureau records: 14.3 %


### 12. Save the feature-engineered dataset

In [167]:
# model_data.to_csv('feature_engineered_data.csv', index=False)

# Feature Engineering Summary

1. Converted raw birth and employment-day variables into interpretable age and employment-duration features.

2. Created affordability and financial-burden features using credit, annuity, income, and family information.

3. Loaded bureau records in chunks and retained all historical credit records corresponding to the sampled application customers.

4. Created indicators representing active loans, closed loans, overdue credit, and outstanding debt.

5. Aggregated the one-to-many bureau table to one row per customer using loan counts, credit amounts, debt amounts, overdue behaviour, and credit-history duration.

6. Created bureau-level ratio features representing active-loan share, overdue-loan share, and debt-to-credit exposure.

7. Merged aggregated bureau features with application data using SK_ID_CURR while preserving all sampled applicants and validating the one-to-one merge.

8. Missing values and categorical encoding were intentionally left for the preprocessing stage to avoid mixing feature engineering with model preparation.

### One important correction before finishing Notebook 3

You should inspect the BUREAU_DEBT_CREDIT_RATIO.

A ratio of:

TOTAL_DEBT / TOTAL_CREDIT

can become unusual if bureau records contain negative debt values, zero total credit, or other anomalies.

In [168]:
bureau_agg[
    [
        'BUREAU_TOTAL_CREDIT',
        'BUREAU_TOTAL_DEBT',
        'BUREAU_DEBT_CREDIT_RATIO'
    ]
].describe(
    percentiles=[0.01, 0.5, 0.95, 0.99]
).T

,count,mean,std,min,1%,50%,95%,99%,max
BUREAU_TOTAL_CREDIT,42851.0,1.965727e+06,3.428846e+06,0.000000e+00,20623.5,962073.000000,7.047740e+06,1.427292e+07,2.146637e+08
BUREAU_TOTAL_DEBT,42851.0,6.466702e+05,1.554879e+06,-2.167229e+06,0.0,166531.500000,2.646317e+06,7.127145e+06,6.508855e+07
BUREAU_DEBT_CREDIT_RATIO,42685.0,2.815107e-01,2.947606e-01,-5.971884e+00,0.0,0.208185,8.572392e-01,9.921950e-01,4.258600e+00


In [169]:
print(
    "Negative Total Credit:",
    (bureau_agg['BUREAU_TOTAL_CREDIT'] < 0).sum()
)

print(
    "Zero Total Credit:",
    (bureau_agg['BUREAU_TOTAL_CREDIT'] == 0).sum()
)

print(
    "Negative Total Debt:",
    (bureau_agg['BUREAU_TOTAL_DEBT'] < 0).sum()
)

print(
    "Debt Ratio > 1:",
    (bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] > 1).sum()
)

Negative Total Credit: 0
Zero Total Credit: 166
Negative Total Debt: 211
Debt Ratio > 1: 252


In [170]:
# Preserve the original aggregated totals.

# Create a valid debt-to-credit ratio only when:
# 1. Total credit is positive
# 2. Total debt is non-negative

valid_ratio_mask = (
    (bureau_agg['BUREAU_TOTAL_CREDIT'] > 0)
    &
    (bureau_agg['BUREAU_TOTAL_DEBT'] >= 0)
)

bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = np.where(
    valid_ratio_mask,
    bureau_agg['BUREAU_TOTAL_DEBT']
        / bureau_agg['BUREAU_TOTAL_CREDIT'],
    np.nan
)

In [171]:
bureau_agg[
    'BUREAU_DEBT_CREDIT_RATIO'
].describe(
    percentiles=[0.01, 0.5, 0.95, 0.99]
)

count    42480.000000
mean         0.283148
std          0.292627
min          0.000000
1%           0.000000
50%          0.210900
95%          0.857912
99%          0.992412
max          4.258600
Name: BUREAU_DEBT_CREDIT_RATIO, dtype: float64

In [172]:
print(
    "Missing Debt-Credit Ratios:",
    bureau_agg[
        'BUREAU_DEBT_CREDIT_RATIO'
    ].isnull().sum()
)

print(
    "Debt Ratio > 1:",
    (
        bureau_agg[
            'BUREAU_DEBT_CREDIT_RATIO'
        ] > 1
    ).sum()
)

print(
    "Negative Debt Ratios:",
    (
        bureau_agg[
            'BUREAU_DEBT_CREDIT_RATIO'
        ] < 0
    ).sum()
)

Missing Debt-Credit Ratios: 371
Debt Ratio > 1: 252
Negative Debt Ratios: 0


In [174]:
model_data = app.merge(
    bureau_agg,
    on='SK_ID_CURR',
    how='left',
    validate='one_to_one'
)

assert len(model_data) == len(app)

model_data.to_csv(
    'feature_engineered_data.csv',
    index=False
)

### Bureau Feature Validation

- Bureau-derived financial features were validated after aggregation.
- 166 customers had zero total recorded credit, making the debt-to-credit ratio undefined.
- 211 customers had negative total debt values, which were considered economically unsuitable for constructing the debt-to-credit ratio.
- The debt-to-credit ratio was therefore calculated only for customers with positive total credit and non-negative total debt.
- Ratios greater than one were retained because they may represent customers whose outstanding debt exceeds their total recorded credit exposure and could contain useful risk information.
- Invalid ratios were represented as missing values and deferred to the preprocessing stage for appropriate treatment.